
# AI Research Math — Master One‑Stop Notebook (Option B)
## Deep Theory • Notation Decoder • Worked Examples • NumPy/PyTorch (Research‑Paper Ready)

This is a **research‑grade** math reference designed to make you fluent at reading AI/ML papers.

It contains:
1) **Symbol & notation decoder** (how to read papers)
2) **Linear algebra** (geometry + computations + tensor shapes)
3) **Set theory & functions** (formal definitions + ML interpretations)
4) **Probability & statistics** (distributions, likelihoods, information theory)
5) **Calculus & matrix calculus** (gradients/Jacobians/Hessians + backprop intuition)
6) **Optimization** (SGD/Adam + conditioning + stability tricks)
7) **Paper equation decoding drills** (including transformer attention)

Every section includes:
- **Notation tables**
- **Theory + intuition**
- **Worked numeric examples**
- **Executable code** in **NumPy** and **PyTorch**



## 0) The 5-step “paper math” decoding workflow

Whenever you see an equation, do this **every time**:

1) **Name the objects:** scalar / vector / matrix / tensor / set / random variable / distribution  
2) **Write shapes:** e.g. $H\in\mathbb{R}^{B\times T\times d}$  
3) **Check operations are legal:** matmul inner dims match; broadcasting; axis reductions  
4) **Translate into words:** “compute similarity → normalize → weighted sum → …”  
5) **Classify the equation:** forward pass / loss / regularizer / gradient / update rule  

This workflow is the difference between “I recognize symbols” and “I can read papers.”



# 1) Symbol & Notation Decoder (with intuition)

## 1.1 Set/logic/definition symbols

| Symbol | Read as | Meaning | Typical paper usage |
|---|---|---|---|
| $x\in A$ | “x in A” | membership | $x\in\mathbb{R}^d$ (x is a d‑vector) |
| $A\subseteq B$ | subset | all elements of A in B | constraint sets |
| $\forall$ | for all | universal | “for all samples” |
| $\exists$ | exists | existence | “there exists a solution” |
| $:=$ | defined as | definition | $z:=Wx+b$ |
| $\approx$ | approx | approximation | variational/approx inference |
| $\propto$ | proportional | up to constant | unnormalized densities |
| $\Rightarrow$ | implies | logical implication | assumptions ⇒ result |
| $\Leftrightarrow$ | iff | equivalence | rewriting conditions |
| $\mathcal{O}(\cdot)$ | big‑O | complexity | attention is $O(T^2)$ |

## 1.2 Core ML symbols

| Symbol | Typical meaning | Notes |
|---|---|---|
| $x$ | input | features/tokens/embeddings (context) |
| $y$ | target | label/next token |
| $\hat y$ | prediction | “y‑hat” |
| $\theta$ | parameters | all weights/biases |
| $W,b$ | weights, bias | linear layer params |
| $B$ | batch size | number of examples |
| $T$ | sequence length | tokens/time steps |
| $d$ | dimension | embedding/model width |
| $\eta$ | learning rate | step size |
| $\lambda$ | regularization | or eigenvalue (context!) |
| $\epsilon$ | small constant/noise | stability, perturbations |
| $\tau$ | temperature | sampling softness |
| $\mathcal{L}$ | loss | objective to minimize |
| $\mathcal{D}$ | dataset distribution | or dataset itself |


In [ ]:

# Quick: keep a Python dict of common symbols (handy for your own notes)
symbol_notes = {
    "∈": "membership: x ∈ A",
    "⊆": "subset: A ⊆ B",
    "∀": "for all",
    "∃": "there exists",
    "∇": "gradient operator",
    "⊙": "elementwise multiply",
    "argmin": "the input value that minimizes a function",
    "~": "distributed as / sampled from",
}
symbol_notes



# 2) Linear Algebra for AI (deep theory + intuition)

Linear algebra is the mathematics of **vector spaces** and **linear maps**.

In AI, it answers:
- What is an **embedding**? (a vector in $\mathbb{R}^d$)
- What does a **layer** do? (a linear map + nonlinearity)
- What is **attention**? (similarity + weighted averaging)
- Why training can be unstable? (conditioning, eigenvalues, singular values)



## 2.1 Scalars, vectors, matrices, tensors (and shapes)

- Scalar: $a\in\mathbb{R}$  
- Vector: $x\in\mathbb{R}^d$  
- Matrix: $W\in\mathbb{R}^{m\times d}$  
- Tensor: $H\in\mathbb{R}^{B\times T\times d}$  

**Interpretation in transformers:**  
$H$ is a batch of sequences of embeddings: batch $B$, tokens $T$, model dimension $d$.


In [ ]:

import numpy as np
import torch

a = 3.0
x = np.array([1.,2.,3.])
W = np.array([[1.,0.,2.],
              [-1.,3.,1.]])
H = torch.randn(2, 4, 8)  # (B,T,d)

a, x.shape, W.shape, tuple(H.shape)



## 2.2 Dot product (inner product)

$$
w^\top x = \sum_{i=1}^d w_i x_i
$$

**Two key intuitions:**
1) **Weighted sum** (a neuron)  
2) **Similarity** (if vectors are normalized)

Cosine similarity:
$$
\cos(\theta)=\frac{x^\top y}{\|x\|_2\|y\|_2}
$$


In [ ]:

w = np.array([0.2, -0.1, 0.5])
x = np.array([1., 2., 3.])
y = np.array([1., 0., 1.])

dot_wx = w @ x
cos_xy = (x @ y) / (np.linalg.norm(x) * np.linalg.norm(y))
dot_wx, cos_xy



## 2.3 Matrix multiplication as “many dot products at once”

If $W\in\mathbb{R}^{m\times d}$ and $x\in\mathbb{R}^d$:
$$
y = Wx \in \mathbb{R}^m,\qquad y_i=\sum_{j=1}^d W_{ij}x_j
$$

**Row view:** each row of $W$ is a weight vector for one output feature.  
**Geometric view:** $W$ is a linear transformation (rotate/scale/shear/project in high‑D).

### Fully worked numeric example
Let
$$
W=\begin{bmatrix}1&2&3\\4&5&6\end{bmatrix},\quad
x=\begin{bmatrix}10\\20\\30\end{bmatrix}
$$
Then
$$
y_1=1\cdot10+2\cdot20+3\cdot30=140,\quad
y_2=4\cdot10+5\cdot20+6\cdot30=320
$$


In [ ]:

W = np.array([[1,2,3],
              [4,5,6]], dtype=float)
x = np.array([10,20,30], dtype=float)
y = W @ x
y


In [ ]:

# Row-by-row dot products (same computation, explicitly)
y1 = W[0] @ x
y2 = W[1] @ x
np.array([y1, y2])



## 2.4 Batch form used in deep learning frameworks

If $X\in\mathbb{R}^{B\times d}$ is a batch and $W\in\mathbb{R}^{m\times d}$, then a common convention:
$$
Y = XW^\top + b,\quad Y\in\mathbb{R}^{B\times m}
$$

This applies the same linear layer to B examples at once.


In [ ]:

X = np.array([[10,20,30],
              [ 1, 2, 3],
              [-1, 0, 1]], dtype=float)  # (B=3,d=3)
W = np.array([[1,2,3],
              [4,5,6]], dtype=float)     # (m=2,d=3)
b = np.array([0.5, -1.0], dtype=float)   # (m,)
Y = X @ W.T + b
X.shape, W.shape, Y.shape, Y


In [ ]:

# PyTorch version
X_t = torch.tensor(X, dtype=torch.float32)
W_t = torch.tensor(W, dtype=torch.float32)
b_t = torch.tensor(b, dtype=torch.float32)
Y_t = X_t @ W_t.T + b_t
Y_t



## 2.5 Elementwise multiplication (Hadamard product) and masks

For same-shape vectors/tensors:
$$
(a\odot b)_i = a_i b_i
$$

Used for:
- gating (LSTM/GRU)
- applying masks (attention masks, dropout masks)
- feature-wise scaling (FiLM, normalization scales)


In [ ]:

a = torch.tensor([1.,2.,3.])
b = torch.tensor([10.,10.,10.])
a * b



## 2.6 Transpose, determinant, eigenvalues, SVD

Transpose:
$$
(A^\top)_{ij}=A_{ji}
$$

Determinant (square A):
- $\det(A)=0$ → not invertible (collapses dimension)  
- $|\det(A)|$ → volume scaling factor

Eigenvalues/eigenvectors:
$$
Av=\lambda v
$$
Directions preserved by the transformation.

SVD:
$$
A=U\Sigma V^\top
$$
Rotate → scale → rotate. Singular values show the strength of independent modes.


In [ ]:

A = np.array([[2.,1.],
              [1.,3.]])
detA = np.linalg.det(A)
eigvals, eigvecs = np.linalg.eig(A)
U, S, Vt = np.linalg.svd(A)
detA, eigvals, S



# 3) Set Theory & Functions (paper-ready)

## 3.1 Dataset notation
A labeled dataset is often written:
$$
\{(x_i,y_i)\}_{i=1}^N
$$
Meaning: for i=1..N, you have an input-output pair.

## 3.2 Function definition
$$
f:X\to Y
$$
For every $x\in X$, there exists exactly one $f(x)\in Y$.

**Important:** output can be a vector/tensor (still one output object).

## 3.3 Parameterized functions (models)
$$
f_\theta(x)
$$
Training chooses $\theta$ to minimize a loss:
$$
\theta^\star=\arg\min_\theta \mathcal{L}(\theta)
$$

## 3.4 Composition (deep networks)
$$
(g\circ f)(x)=g(f(x))
$$

## 3.5 Deterministic vs stochastic modeling
Deterministic: $y=f_\theta(x)$  
Stochastic: $y\sim p_\theta(y\mid x)$ (model outputs a distribution; sampling draws outputs)


In [ ]:

# Dataset notation in Python
N = 5
xs = np.arange(N)
ys = xs**2
dataset = list(zip(xs, ys))
dataset


In [ ]:

# Parameterized function (linear model): y = Wx + b
def f_theta(x, W, b):
    return W @ x + b

x = np.array([1.,2.,3.])
W = np.array([[1.,0.,2.],
              [-1.,3.,1.]])
b = np.array([0.5, -1.0])

f_theta(x, W, b)



# 4) Probability & Statistics (AI essentials)

Probability appears because models often predict **distributions**, not single values.

## 4.1 Random variables and distributions
- Discrete: PMF $P(X=x)$  
- Continuous: PDF $p(x)$ with probabilities via integrals:
$$
P(a\le X\le b)=\int_a^b p(x)\,dx
$$

Notation:
- $X\sim \mathcal{N}(\mu,\sigma^2)$ (X distributed as a normal)
- $x\sim p(x)$ (x sampled from p)

## 4.2 Expectation and variance
$$
\mu=\mathbb{E}[X],\quad \mathrm{Var}(X)=\mathbb{E}[(X-\mu)^2],\quad \sigma=\sqrt{\mathrm{Var}(X)}
$$

## 4.3 Mean/variance from data (statistics)
For samples $x_1,\dots,x_n$:
$$
\bar{x}=\frac1n\sum_{i=1}^n x_i
$$

## 4.4 Information theory (paper staples)
Entropy:
$$
H(P)=-\sum_x P(x)\log P(x)
$$
Cross-entropy:
$$
H(P,Q)=-\sum_x P(x)\log Q(x)
$$
KL divergence:
$$
D_{KL}(P\|Q)=\sum_x P(x)\log\frac{P(x)}{Q(x)}
$$

## 4.5 Softmax and categorical distributions (language modeling)
Given logits $z\in\mathbb{R}^V$:
$$
p=\mathrm{softmax}(z)
$$
Temperature:
$$
p=\mathrm{softmax}(z/\tau)
$$


In [ ]:

data = np.array([2.,4.,4.,4.,5.,5.,7.,9.])
mean = data.mean()
var = ((data-mean)**2).mean()
std = var**0.5
mean, var, std


In [ ]:

import torch
torch.manual_seed(0)

logits = torch.tensor([2.0, 1.0, 0.1])
probs = torch.softmax(logits, dim=0)
samples = torch.multinomial(probs, num_samples=10, replacement=True)
probs, samples


In [ ]:

# Cross-entropy in PyTorch (classification / next-token style)
logits = torch.tensor([[2.0, 1.0, 0.1]])
true_class = torch.tensor([0])
ce = torch.nn.functional.cross_entropy(logits, true_class)
ce.item()


In [ ]:

# KL divergence example (categorical)
p = torch.tensor([0.7, 0.2, 0.1])
q = torch.tensor([0.6, 0.3, 0.1])
kl = (p * (p.log() - q.log())).sum()
kl.item()



# 5) Calculus & Matrix Calculus (for backprop)

## 5.1 Derivative (1D)
$$
f'(x)=\lim_{h\to 0}\frac{f(x+h)-f(x)}{h}
$$

## 5.2 Gradient (multi-D)
For $f:\mathbb{R}^n\to\mathbb{R}$:
$$
\nabla_x f(x)=\begin{bmatrix}\partial f/\partial x_1\\ \vdots\\ \partial f/\partial x_n\end{bmatrix}
$$
**Intuition:** direction of steepest increase.

## 5.3 Jacobian (vector output)
For $g:\mathbb{R}^n\to\mathbb{R}^m$:
$$
(J_g(x))_{ij}=\frac{\partial g_i}{\partial x_j}
$$

## 5.4 Chain rule (backprop)
If $y=f(u)$ and $u=g(x)$:
$$
\frac{dy}{dx}=\frac{dy}{du}\frac{du}{dx}
$$

## 5.5 Gradients w.r.t. parameters
If $\theta$ is a parameter vector, $\nabla_\theta \mathcal{L}$ has the same “shape” as $\theta$.


In [ ]:

# Finite difference derivative check
def f(x): return x**2
x = 3.0
h = 1e-6
fd = (f(x+h)-f(x))/h
fd, 2*x


In [ ]:

# Autograd gradients (PyTorch)
import torch
torch.manual_seed(0)

x = torch.tensor([1.0, -2.0, 3.0], requires_grad=True)
L = (x**2).sum() + 2*x[0]*x[1]
L.backward()
L.item(), x.grad


In [ ]:

# Jacobian (PyTorch)
x = torch.tensor([1.0, 2.0], requires_grad=True)

def g(x):
    return torch.stack([x[0]**2 + x[1], torch.sin(x[0]) + x[1]**3])

J = torch.autograd.functional.jacobian(g, x)
g(x).detach(), J.detach()



# 6) Optimization (how training updates happen)

Most training is written:
$$
\theta^\star=\arg\min_\theta \mathcal{L}(\theta)
$$

## 6.1 Gradient descent
$$
\theta \leftarrow \theta - \eta \nabla_\theta \mathcal{L}(\theta)
$$

## 6.2 Stochastic gradient descent (SGD)
Uses mini-batches; gradient is a noisy estimate.

## 6.3 Adam (paper recognition)
Papers use $m_t, v_t, \hat m_t, \hat v_t$ for first/second moment estimates.

## 6.4 Numerical stability tricks you’ll see
- stable softmax: subtract $\max(z)$  
- log-sum-exp trick  
- add $\epsilon$ inside $\log$ or denominator to avoid divide-by-zero


In [ ]:

# Simple optimization demo in PyTorch: fit y ≈ w x + b
import torch
torch.manual_seed(0)

X = torch.linspace(-1, 1, 100).unsqueeze(1)
true_w, true_b = 2.0, -0.5
y = true_w*X + true_b + 0.1*torch.randn_like(X)

w = torch.randn(1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

opt = torch.optim.Adam([w, b], lr=0.1)

for _ in range(300):
    opt.zero_grad()
    yhat = w*X + b
    loss = ((yhat - y)**2).mean()
    loss.backward()
    opt.step()

w.item(), b.item(), loss.item()


In [ ]:

# Stable softmax demo (NumPy)
import numpy as np

z = np.array([1000.0, 1001.0, 999.0])
stable = np.exp(z - z.max()) / np.exp(z - z.max()).sum()
stable



# 7) Paper Equation Decoding Drills (practice reading)

Below are common “paper equations” and how to read them.

## 7.1 Cross-entropy (classification / next-token)
$$
\mathcal{L} = -\sum_{i=1}^V y_i \log p_i,\quad p=\mathrm{softmax}(z)
$$
Read: “convert logits z to probabilities p; penalize log-prob of the true label.”

## 7.2 L2 regularization / weight decay
$$
\mathcal{L}_{total}=\mathcal{L}+\lambda\|\theta\|_2^2
$$
Read: “add a penalty for large weights.”

## 7.3 Attention (transformers)
$$
\mathrm{Attn}(Q,K,V)=\mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$
Read:
1) $QK^\top$ → token-to-token similarity scores  
2) scale by $\sqrt{d_k}$  
3) softmax → attention weights (rows sum to 1)  
4) weights × V → weighted sums of value vectors  


In [ ]:

# Mini attention-shaped computation (shapes + real numbers)
import torch
torch.manual_seed(0)

B, T, d = 2, 4, 8
H = torch.randn(B, T, d)

WQ = torch.randn(d, d)
WK = torch.randn(d, d)
WV = torch.randn(d, d)

Q = H @ WQ
K = H @ WK
V = H @ WV

scores = (Q @ K.transpose(1,2)) / (d ** 0.5)   # (B,T,T)
weights = torch.softmax(scores, dim=-1)        # (B,T,T) rows sum to 1
out = weights @ V                               # (B,T,d)

scores.shape, weights.shape, out.shape, weights[0,0].sum().item()
